# CLIQUE Customer Clustering — Training

Online Retail II → profiles → CLIQUE + baselines.

In [ ]:
# 1. Install / import dependencies
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from clique.algorithm import CLIQUE
from clique.utils import (
    FEATURE_NAMES,
    build_customer_profiles,
    clean_data,
    compute_silhouette,
    load_raw_data,
    preprocess,
    run_baseline_comparison,
    save_model,
)

sns.set_theme(style="whitegrid")
MODELS_DIR = ROOT / "models"
DATA_PATH = ROOT / "data" / "online_retail_ii.xlsx"
print("ROOT:", ROOT)

In [ ]:
# 2. Load and clean Online Retail II
if not DATA_PATH.exists():
    print("Dataset not found. Download from:")
    print("https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip")
    print("Place online_retail_ii.xlsx in data/ — using mock data fallback.")
    mock = pd.read_csv(ROOT / "data" / "test_clean.csv")
    customer_profiles = mock
    cancellations_df = pd.DataFrame()
else:
    raw = load_raw_data(str(DATA_PATH))
    clean_df, cancellations_df = clean_data(raw, filter_uk=True)
    customer_profiles = build_customer_profiles(clean_df, cancellations_df)
customer_profiles.head()

In [ ]:
# 3. EDA: histogram, boxplot, correlation heatmap
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes.ravel(), FEATURE_NAMES):
    customer_profiles[col].hist(ax=ax, bins=30)
    ax.set_title(col)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.boxplot(data=customer_profiles[FEATURE_NAMES], orient="h")
plt.title("Feature boxplots")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(customer_profiles[FEATURE_NAMES].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature correlation")
plt.show()

In [ ]:
# 4–7. Preprocess: log-transform, split, MinMaxScaler
X_train, X_test, ids_train, ids_test, scaler = preprocess(
    customer_profiles, test_size=0.2, random_state=42, models_dir=str(MODELS_DIR)
)

In [ ]:
# 8. CLIQUE grid search
grid_rows = []
for xi in [5, 8, 10]:
    for tau in [0.02, 0.05, 0.08]:
        m = CLIQUE(xi=xi, tau=tau)
        m.fit(X_train, feature_names=FEATURE_NAMES)
        cov = np.mean(list(m.subspace_coverage_.values())) if m.subspace_coverage_ else 0
        sil = compute_silhouette(X_train, m.labels_)
        grid_rows.append({
            "xi": xi, "tau": tau,
            "n_clusters": len(m.clusters_),
            "coverage": cov, "silhouette": sil,
        })
grid_df = pd.DataFrame(grid_rows)
print(grid_df.sort_values("silhouette", ascending=False).to_string(index=False))
best = grid_df.loc[grid_df["silhouette"].idxmax()]
best_xi, best_tau = int(best["xi"]), float(best["tau"])
print(f"Best params: xi={best_xi}, tau={best_tau}")

In [ ]:
# 9. Train final CLIQUE and save artifacts
final_model = CLIQUE(xi=best_xi, tau=best_tau)
final_model.fit(X_train, feature_names=FEATURE_NAMES)
save_model(final_model, scaler, final_model.cluster_profiles_, str(MODELS_DIR))
print(f"Saved {len(final_model.clusters_)} clusters to {MODELS_DIR}")

In [ ]:
# 10. Baseline comparison
baseline_df = run_baseline_comparison(X_train)
display(baseline_df)

In [ ]:
# 11. Grid visualization for top 2 subspaces
ranked = sorted(final_model.subspace_coverage_.items(), key=lambda x: -x[1])
for subspace, _ in ranked[:2]:
    if len(subspace) >= 2:
        d1, d2 = subspace[0], subspace[1]
        fig = final_model.plot_grid_static(X_train, d1, d2, FEATURE_NAMES)
        fig.show()